[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/spherical_quadrics/spherical_quadrics.ipynb)

# A Conic on the Sphere

This example constructs a spherical ellipse on the unit sphere and verifies its focal sum, tangent envelope, and level sets. A quadratic cone through the sphere's centre cuts the surface in a closed oval whose geodesic distances along great circles to two foci sum to twice the semi-major arc. Read through its tangent great circles instead of its points, the same curve satisfies a dual focal law in which the product of the sines of the two foci's distances to every tangent great circle is constant. Evaluated across the whole sphere, the cone's quadratic form traces Poinsot's polhodes as its level sets.

In 3D spherical geometric algebra, points on the sphere are modelled as unit antivectors and planes through the centre (great circles) as unit vectors. The conic is a polarity map `conic` from points to polar planes built as a weighted dyad sum `(planes * (planes & Point) * eigenvalues).sum(axis=0)`, with the curve at `conic(point) & point == 0`. Applying `conic` to points along the oval gives its tangent great circles directly, while solving the principal map `principal.solve(rotor << Plane)` forms the dual conic on planes, and one rotor transports both polarities by `rotor >> principal(rotor << Point)`.

In 3D Cartesian matrix and rigid-body notation, the cone is written as $\mathbf{x}^\top R\,\operatorname{diag}(\lambda_1, \lambda_2, \lambda_3)\,R^\top \mathbf{x} = 0$ on $\|\mathbf{x}\| = 1$, tangent great-circle normals require normalizing $C\mathbf{x} / \|C\mathbf{x}\|$ and inverting $C^{-1}$ separately for the dual cone $\mathbf{n}^\top C^{-1} \mathbf{n} = 0$, geodesic arcs and point-to-circle sines split into $\arccos(\mathbf{x}\cdot\mathbf{f}_i)$ and $|\mathbf{n}\cdot\mathbf{f}_i|$, and Poinsot's polhodes appear as intersections of the kinetic-energy ellipsoid $\boldsymbol{\omega}^\top I \boldsymbol{\omega} = 2E$ with the angular-momentum sphere $\|I\boldsymbol{\omega}\|^2 = L^2$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import Spherical3D as ga
from examples.util.animation import save_figure
from examples.quadrics.spherical_quadrics import render, scenarios

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# Points lie on yz, zx, xy as the poles of the coordinate planes x, y, z.
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.plane()                           # Vector+
Rotor = ga.gatype.rotor()                           # Rotor+
Polarity = ga.gatype((Plane, Point))
DualPolarity = ga.gatype((Point, Plane))

## Principal polarity, oval locus, and dual conic

With two positive eigenvalues and one negative eigenvalue on the coordinate planes, the principal polarity defines a real cone around the negative-eigenvalue pole that meets the unit sphere in a closed oval. Meridian rays from that pole toward points around the equator determine the cone's slope directly from the ratio of their quadratic forms, and a single rotor transports the polarity, its dual inverse on planes, the oval, and both foci into position.

In [ ]:
eigenvalues = scenarios.eigenvalues
turn_xy, turn_yz = scenarios.turn_xy, scenarios.turn_yz
n_oval = scenarios.n_oval
sample_index = scenarios.sample_index

In [ ]:
def principal_polarity(eigenvalues: np.ndarray) -> Polarity:
    """Weighted dyads of the coordinate planes form the diagonal polarity (`[] Plane+ <- Point-`)."""
    planes = stack([mv.x, mv.y, mv.z])                                          # [3] Plane+
    return (planes * (planes & Point) * eigenvalues).sum(axis=0)                # [] Plane+ <- Point-


def conic_locus(principal: Polarity, azimuths: np.ndarray) -> tuple[Point, Point, Scalar]:
    """Meridian rays around the negative-eigenvalue pole trace the spherical oval (`[...] Point-`), its two foci (`[2] Point-`), and its semi-major arc (`[] Scalar+`)."""
    pole = mv.z.dual()                                                          # [] Point-
    meridian_turns = (mv.xy * (-azimuths / 2)).exp()                            # [...] Rotor+
    equator = meridian_turns >> mv.x.dual()                                     # [...] Point-
    # Along `pole + equator * slopes`, cross terms vanish on principal axes so the cone cancels the pole and equator forms.
    slopes = (-(principal(pole) & pole) / (principal(equator) & equator)).square_root()  # [...] Scalar+
    oval_points = (pole + equator * slopes).normalized().retyped(Point)         # [...] Point-

    # The minor and major vertices lie toward the x and y poles; their cosine ratio sets the focal arc along the major meridian.
    equator_axes = stack([mv.x, mv.y]).dual()                                   # [2] Point-
    axis_slopes = (-(principal(pole) & pole) / (principal(equator_axes) & equator_axes)).square_root()  # [2] Scalar+
    minor_vertex, major_vertex = (pole + equator_axes * axis_slopes).normalized()  # [] Point-, [] Point-
    semi_major = (pole.reverse() | major_vertex).arccos()                       # [] Scalar+
    focal_arc = ((pole.reverse() | major_vertex) / (pole.reverse() | minor_vertex)).arccos()  # [] Scalar+
    focal_turns = (mv.yz * (stack([focal_arc, -focal_arc]) / 2)).exp()          # [2] Rotor+
    foci = (focal_turns >> pole).retyped(Point)                                 # [2] Point-
    return oval_points, foci, semi_major


rotor = (mv.xy * turn_xy + mv.yz * turn_yz).exp()                               # [] Rotor+
principal = principal_polarity(eigenvalues)                                     # [] Plane+ <- Point-
conic = rotor >> principal(rotor << Point)                                      # [] Plane+ <- Point-
dual = rotor >> principal.solve(rotor << Plane)                                 # [] Point- <- Plane+
rest_oval, rest_foci, semi_major = conic_locus(principal, np.linspace(0, 2 * np.pi, n_oval))
oval_points = (rotor >> rest_oval).retyped(Point)                               # [n_oval] Point-
foci = (rotor >> rest_foci).retyped(Point)                                      # [2] Point-

## Focal geodesic sum and dual tangent product

The geodesic distance between two unit points on the sphere has cosine `point.reverse() | focus`, and summing the two focal arcs at every point of the oval yields twice the semi-major arc. Evaluating `conic` on the oval gives its unit tangent great circles, and pairing each tangent plane with the two foci yields sines whose product stays constant around the envelope.

In [ ]:
focal_sum = (oval_points[:, None].reverse() | foci).clip(-1.0, 1.0).arccos().sum(axis=-1)  # [n_oval] Scalar+
tangents = conic(oval_points).normalized()                                      # [n_oval] Plane+
dual_product = (tangents[:, None] & foci).abs().cumprod(axis=-1)[:, -1]         # [n_oval] Scalar+
sample_point = oval_points[sample_index]                                        # [] Point-

print("sum of focal distances:", focal_sum.mean().to_array(), "  twice semi-major arc:", (2 * semi_major).to_array())
print("product of focal sines to tangents:", dual_product.mean().to_array())

save_figure(render.conic_figure(oval_points, foci, sample_point, tangents, conic), "spherical_quadrics");

The left panel shades the quadratic form `conic(point) & point` from blue below zero through neutral white to red above zero, with the spherical oval in gold, its antipodal twin dashed on the opposite hemisphere, the two foci in red, and the lime and cyan geodesic arcs meeting at the white sample point. The middle panel draws the oval as the envelope of plasma-coloured tangent great circles, with short arrows turned toward their poles and magenta dots tracing the dual conic along those poles. The right panel shows the khaki quadratic cone slicing the sphere along the gold oval together with salmon and sky-blue level curves (Poinsot's polhodes) below and above zero.

In [ ]:
# checks
assert ((conic(oval_points) & oval_points).abs() < 1e-12).all()
assert ((focal_sum - 2 * semi_major).abs() < 1e-11).all()
assert ((tangents & dual(tangents)).abs() < 1e-12).all()
assert ((dual_product - dual_product.mean()).abs() < 1e-13).all()